# Strategy ladder rung 4 on curated-label questions


**What this tests (instruction 73, phase 4).** Whether a model per evidence family combined (rung 2) or network propagation (rung 3) answers the curated-label questions better than boosted trees on all the evidence at once (rung 1), and whether more cells become fillable at the 90 % bar.

**Fair comparison:** the inner-fold score of every strategy on the same folds (average precision, the harness selection score). The frozen test is touched once, for the strategy the inner folds chose.

Built 2026-11-08.

In [1]:
import glob, json, os, numpy as np, pandas as pd
recs = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(REPO, SCORES, '*.json')))]
R = pd.DataFrame([r for r in recs if 'error' not in r])
errors = [r for r in recs if 'error' in r]
rung1 = pd.read_csv(os.path.join(REPO, 'results/question_scoreboard_2026_11_08/scores.tsv'), sep='\t').set_index('question')
print(len(R), 'questions scored;', len(errors), 'errors')
R.groupby('category').size().to_frame('questions')

69 questions scored; 0 errors


,questions
category,
function,2
interaction,12
modification,22
phenotype,6
place,27


## 1 · Same folds: which strategy wins the inner folds

In [2]:
rows = []
for r in R.itertuples(index=False):
    folds = pd.DataFrame(r.inner_scores).T
    rows.append(dict(question=r.question, category=r.category, **folds.mean().round(4).to_dict()))
inner = pd.DataFrame(rows)
strats = [c for c in inner.columns if c not in ('question', 'category')]
inner['best'] = inner[strats].fillna(-1).idxmax(axis=1).where(inner[strats].notna().any(axis=1), '')
inner['gain_over_rung1'] = (inner[strats].max(axis=1) - inner['boosted trees']).round(4)
inner.to_csv(os.path.join(OUT, 'inner_scores.tsv'), sep='\t', index=False)
summary = inner.groupby('category')[strats].median().round(3)
summary['questions'] = inner.groupby('category').size()
summary['rung 2 or 3 best'] = inner[inner.best != 'boosted trees'].groupby('category').size().reindex(summary.index).fillna(0).astype(int)
summary

,boosted trees,low_rank_completion,questions,rung 2 or 3 best
category,,,,
function,0.326,0.265,2,1
interaction,0.348,0.364,12,5
modification,0.392,0.333,22,7
phenotype,0.256,0.295,6,4
place,0.494,0.344,27,0


## 2 · Frozen test of the chosen strategy, and cells fillable at 90 %

Rung 1 numbers are from seed 0, the ladder from seed 2: different partitions.

In [3]:
R['rung1_frozen_auc'] = R.question.map(rung1.frozen_auc)
R['rung1_may_fill'] = R.question.map(rung1.may_fill)
R['chosen_mostly'] = R.chosen.map(lambda c: pd.Series(c).mode().iloc[0] if len(c) else '')
cols = ['question', 'category', 'chosen_mostly', 'frozen_auc', 'rung1_frozen_auc', 'precision_lower90', 'may_fill', 'rung1_may_fill']
F = R[cols].sort_values('frozen_auc', ascending=False)
F.to_csv(os.path.join(OUT, 'frozen.tsv'), sep='\t', index=False)
out = dict(questions=len(R), errors=len(errors),
           median_frozen_auc=round(float(R.frozen_auc.median()), 3),
           median_rung1_frozen_auc=round(float(R.rung1_frozen_auc.median()), 3),
           may_fill=int(R.may_fill.sum()), rung1_may_fill=int(R.rung1_may_fill.fillna(False).sum()),
           inner_best=inner.best.value_counts().to_dict())
json.dump(out, open(os.path.join(OUT, 'summary.json'), 'w'), indent=1)
print(out)
F.head(20).round(3)

{'questions': 69, 'errors': 0, 'median_frozen_auc': 0.777, 'median_rung1_frozen_auc': 0.817, 'may_fill': 4, 'rung1_may_fill': 2, 'inner_best': {'boosted trees': 52, 'low_rank_completion': 16, '': 1}}


,question,category,chosen_mostly,frozen_auc,rung1_frozen_auc,precision_lower90,may_fill,rung1_may_fill
29,cat_lopit_unified_ribosome_RNP,place,boosted trees,1,0.945,0.614,False,False
14,cat_compartment_rhoptries_1,place,boosted trees,0.996,0.999,0.416,False,False
15,cat_compartment_rhoptries_2,place,boosted trees,0.989,0.986,NaN,False,False
8,cat_compartment_dense_granules,place,boosted trees,0.983,0.947,0.69,False,False
56,cat_pvm_proximity_positive_1_0,place,boosted trees,0.975,1,0.774,False,False
16,cat_compartment_unassigned,place,boosted trees,0.974,0.974,0.897,False,False
23,cat_lopit_unified_apicoplast,place,boosted trees,0.962,0.955,0.316,False,False
22,cat_lopit_unified_apical_secretory,place,boosted trees,0.958,0.934,0.901,True,False
7,cat_compartment_cytosol,place,boosted trees,0.958,0.949,0.767,False,False
9,cat_compartment_micronemes,place,boosted trees,0.958,0.999,NaN,False,False


## 3 · Reading

A strategy that wins the inner folds on a category is the one to use there; a category where boosted trees still win shows that splitting evidence by family or walking the evidence graph adds nothing on these labels. Cells become fillable only where the frozen lower bound reaches 0.90.